# 🎯 ใบงานสัปดาห์ที่ 02 — ประเภทของ Machine Learning และการกำหนดตัวแปรเป้าหมาย

---

**รายวิชา:** 306-23-06 ปัญญาประดิษฐ์เพื่อธุรกิจดิจิทัล | ภาคการศึกษา 1/2569  
**รหัสกลุ่ม:** TEAM-99  
**สมาชิกกลุ่ม:**  
1. น.ส.กมลวรรณ จันทร์ผึ้ง (001)  
2. น.ส.ชลดา อิศรเสนา ณ อยุธยา (009)  
3. น.ส.ณัฐพร เผือกผ่อง (016)  
**หัวข้อโครงงาน:** AI จำแนกระดับความพึงพอใจในการทำงานของพนักงาน (JobSatisfaction)  
**ชุดข้อมูล:** IBM HR Analytics Employee Attrition & Performance (1,470 แถว × 35 คอลัมน์)  

---

### 📌 วัตถุประสงค์และภาพรวม
วิเคราะห์ประเภทปัญหาทางปัญญาประดิษฐ์ กำหนดตัวแปรเป้าหมาย (Target) และฟีเจอร์ตั้งต้น (Features) สำหรับการจำแนกระดับความพึงพอใจของพนักงาน


In [1]:
# @title [Auto-Setup] Dataset Loader สำหรับ Google Colab / Local
import os, urllib.request, pandas as pd, shutil

DATA_URL = 'https://raw.githubusercontent.com/ibm-watson-data-lab/open-data/master/HR-Employee-Attrition.csv'

# 1. เตรียม Raw Dataset
os.makedirs('data', exist_ok=True)
if not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    try:
        print('Downloading raw dataset...')
        urllib.request.urlretrieve(DATA_URL, 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
    except Exception as e:
        print('Note:', e)

if os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('WA_Fn-UseC_-HR-Employee-Attrition.csv', 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
elif os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('data/WA_Fn-UseC_-HR-Employee-Attrition.csv', 'WA_Fn-UseC_-HR-Employee-Attrition.csv')

# 2. เตรียม clean.csv / clean (4).csv สำหรับโมเดล
if not os.path.exists('clean.csv') and not os.path.exists('clean (4).csv'):
    raw_path = 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv' if os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') else 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
    if os.path.exists(raw_path):
        raw_df = pd.read_csv(raw_path)
        cols_to_drop = [c for c in ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'] if c in raw_df.columns]
        clean_df = raw_df.drop(columns=cols_to_drop)
        num_cols = clean_df.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = clean_df.select_dtypes(include=['object']).columns
        clean_df = pd.get_dummies(clean_df, columns=cat_cols, drop_first=True)
        clean_df.to_csv('clean.csv', index=False)

if os.path.exists('clean.csv'):
    if not os.path.exists('clean (4).csv'): shutil.copyfile('clean.csv', 'clean (4).csv')
    if not os.path.exists('clean (3).csv'): shutil.copyfile('clean.csv', 'clean (3).csv')

print(' Dataset ready for execution!')


In [9]:
# Dataset พร้อมใช้งานผ่าน Auto-Setup แล้ว
# from google.colab import files
# files.upload()
print("✓ ใช้ข้อมูลจากระบบ Auto-Setup เรียบร้อยแล้ว")


✓ ใช้ข้อมูลจากระบบ Auto-Setup เรียบร้อยแล้ว


In [11]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [12]:
!kaggle datasets download -d pavansubhasht/ibm-hr-analytics-attrition-dataset --unzip -p data/


Dataset URL: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
License(s): DbCL-1.0
100% 50.1k/50.1k [00:00<00:00, 56.5MB/s]



In [17]:
import pandas as pd
hr_data = pd.read_csv("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(hr_data["JobSatisfaction"].dtype)
hr_data["JobSatisfaction"].value_counts()

int64


,count
JobSatisfaction,
4,459
3,442
1,289
2,280


In [16]:
hr_data["JobSatisfaction"].nunique()

4

In [ ]:
hr_data = pd.read_csv("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(hr_data["JobSatisfaction"].dtype)
hr_data["JobSatisfaction"].describe()

object


,Attrition
count,1470
unique,2
top,No
freq,1233


In [15]:
hr_data["JobSatisfaction"].nunique()

4

---

## 📝 สรุปผลและตอบคำถามวัดความเข้าใจตามใบงาน

### สรุปคำถามและการวิเคราะห์:
1. **ประเภทของ Machine Learning:** เป็น **Supervised Learning** ประเภท **Multi-class Classification** เพราะมีตัวแปรเป้าหมายที่ทราบค่าล่วงหน้า และเป้าหมายแบ่งออกเป็น 4 ระดับ (1=พอใจน้อย, 2=ปานกลาง, 3=พอใจมาก, 4=พอใจมากที่สุด)
2. **ความสมดุลของคลาส:** คลาส 1 (20%), คลาส 2 (19%), คลาส 3 (30%), คลาส 4 (31%) จัดว่ามีการกระจายตัวที่สมดุลดี ไม่จำเป็นต้องทำ Over-sampling
